In [2]:
%%writefile zoo.py
def hours():
    print('Open 9-5 daily')

Writing zoo.py


In [5]:
import zoo
zoo.hours()

Open 9-5 daily


In [6]:
import zoo as menagerie
menagerie.hours()

Open 9-5 daily


In [23]:
# Clear all rows from books table since it was printing titles multipule times due to running 16.5 multiple times
import sqlite3

conn = sqlite3.connect("books.db")
cursor = conn.cursor()

cursor.execute("DELETE FROM books")

conn.commit()
conn.close()


In [24]:
%%writefile books2.csv
title,author,year
The Weirdstone of Brisingamen,Alan Garner,1960
Perdido Street Station,China Miéville,2000
Thud!,Terry Pratchett,2005
The Spellman Files,Lisa Lutz,2007
Small Gods,Terry Pratchett,1992

Overwriting books2.csv


In [25]:
# 16.4 — create SQLite database and table
import sqlite3
# creates and connects the database
conn = sqlite3.connect("books.db")
cursor = conn.cursor()
# creates table
cursor.execute("""
CREATE TABLE IF NOT EXISTS books (
    title TEXT,
    author TEXT,
    year INTEGER
)
""")
# commit changes and closes the connection
conn.commit()
conn.close()

In [26]:
# 16.5 — Insert CSV data into the books table.
# csv.reader() loads each row as a list of strings.
# next(reader) skips the header row so only data rows are inserted.

import csv
import sqlite3

# Connect to the existing database.
conn = sqlite3.connect("books.db")
cursor = conn.cursor()

# Open the CSV file and read its contents.
with open("books2.csv", newline="") as f:
    reader = csv.reader(f)
    next(reader)  # Skip header row: title, author, year

    # Insert each row into the database.
    for title, author, year in reader:
        cursor.execute(
            "INSERT INTO books VALUES (?, ?, ?)",
            (title, author, int(year))  # Convert year to integer
        )

# Commits changes and closes the connection.
conn.commit()
conn.close()

In [27]:
# 16.8 — Use SQLAlchemy to connect to the SQLite database and query data.
# SQLAlchemy provides a higher-level interface than sqlite3.

from sqlalchemy import create_engine, MetaData, select

# Create engine pointing to the same books.db file.
engine = create_engine("sqlite:///books.db")

# Reflect existing database structure into SQLAlchemy metadata.
metadata = MetaData()
metadata.reflect(bind=engine)

# Accesses the books table.
books = metadata.tables["books"]

# Build a SELECT query for the title column, sorted alphabetically.
query = select(books.c.title).order_by(books.c.title)

# Executes the query and prints each title.
with engine.connect() as conn:
    results = conn.execute(query)
    for row in results:
        print(row.title)

Perdido Street Station
Small Gods
The Spellman Files
The Weirdstone of Brisingamen
Thud!
